# 13 csakd yolo26s to yolo26n — Google Colab

1. Set **Runtime > Change runtime type > GPU** for training.
2. Edit `DATASET_ZIP` and `DRIVE_OUTPUT_ROOT` below.
3. Add `WANDB_API_KEY` in **Colab Secrets** and enable notebook access (training only).
4. Run All. Research hyperparameters and seed schedules are in the next configuration cell.

Source: `12_csakd_yolo26n_teacher_yolo26s.ipynb`. Experiment IDs/artifact names are preserved. This copy is standalone; no repository checkout is required.

### Method notes

**Eksperimen 12** · Adaptasi CSAKD enam modul untuk transfer YOLO26s → YOLO26n, dengan Cross Supervising, AAWM, dan AGKA.

Protokol penelitian: Poltrada V3, 100 epoch, imgsz640, batch16, seed42, AdamW lr0.001, AMP aktif. Teacher adalah best.pt dari artifact baseline YOLO26s **v62** yang ditetapkan di section1, sama dengan notebook 10–11; nomor versi artifact tidak menyatakan nomor epoch.

Notebook ini merupakan **adaptasi penelitian**, bukan reproduksi identik self-distillation YOLOv8 dan hybrid quantization dalam paper. Perbedaan formulasi paper/kode penulis dan batas pengujian dijelaskan di section5 dan laporan audit. Bobot AGKA/Cross 1/1 adalah pilihan adaptasi; belum diklaim optimal.

Run All langsung menjalankan target 100 epoch setelah verifikasi teacher. Evaluasi akhir menggunakan **validation (val)**; test tidak digunakan untuk tuning atau memilih checkpoint. Artifact per epoch menyediakan student-only best.pt/last.pt untuk inferensi, serta training_state_* asli yang memerlukan definisi class notebook. Resume CSAKD tidak didukung.


## Colab paths

Keep the ZIP unchanged in Drive. Extraction and the derived YAML are local; source images, labels and YAML are not edited. For multiple YAML files, set `DATASET_YAML_RELATIVE` to the desired archive-relative path.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

# Edit these two paths to match your Google Drive.
DATASET_ZIP = "/content/drive/MyDrive/Skripsi/datasets/poltrada-v3-yolo26-clean.zip"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/Skripsi/yolo26-kd-indotraffic/results"
DATASET_YAML_RELATIVE = ""  # Set only if the ZIP contains more than one YAML.
SOURCE_DATA_DECLARATION = '/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml'  # Original teacher provenance; not an active Colab path.
COLAB_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
COLAB_LOCAL_DIR = Path("/content/yolo26_colab") / COLAB_RUN_ID
COLAB_OUTPUT_DIR = Path(DRIVE_OUTPUT_ROOT) / "13_csakd_yolo26s_to_yolo26n" / COLAB_RUN_ID
COLAB_DATA = str(COLAB_LOCAL_DIR / "runtime/data.yaml")


## Research configuration

Preserved from the original notebook. The trio defaults to 100 epochs with smoke disabled. Existing multi-seed queues remain unchanged; 100 epochs is per seed.

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "12_csakd_yolo26n_teacher_yolo26s"
METHOD = "csakd"
DATA = COLAB_DATA  # Runtime YAML prepared from the Drive ZIP.
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0                   # GPU Colab; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # validation; test tidak dipakai untuk pemilihan checkpoint
PROJECT = str(COLAB_OUTPUT_DIR)  # Persistent Google Drive output for this run.
TEACHER_CHECKPOINT = ""   # Diisi otomatis dari artifact baseline YOLO26s W&B v62 di section 4.
KD_AGKA_WEIGHT = 1.0
KD_CROSS_WEIGHT = 1.0
KD_CHECKPOINT = True        # Recompute aktivasi KD saat backward; batch student tetap BATCH.
KD_CHUNK_SIZE = 2           # Chunk hanya projector/cross/attention/loss, bukan student forward.
EXPECTED_NC = 6
TEACHER_ARTIFACT = (
    "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/"
    "02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
)
TEACHER_RUN_ID = "q5kp9k9d"
CSAKD_AUTHOR_COMMIT = "3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8"
DIAGNOSTIC_ONLY = False      # True hanya untuk fixture/smoke; bukan hasil penelitian.

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"

WANDB_RUN_NAME = f"csakd_T26s_S26n_seed{SEED}"

# Group membandingkan pasangan yang sama; tag mengikuti konfigurasi run baru.
WANDB_GROUP = "T26s_S26n"
WANDB_TAGS = [
    METHOD, 'teacher-s', 'student-n', f"agka{KD_AGKA_WEIGHT:g}", f"cross{KD_CROSS_WEIGHT:g}", f"seed{SEED}", "poltrada-v3", EXPERIMENT_ID
]


## Colab: Drive, dependencies, and local dataset

Train from the local extracted ZIP; outputs go to Drive and existing W&B callbacks. Authorize Drive mounting when prompted.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
COLAB_LOCAL_DIR.mkdir(parents=True, exist_ok=False)
COLAB_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(COLAB_LOCAL_DIR)  # Downloads and temporary files stay on the Colab VM.
os.environ["WANDB_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "wandb-cache")
os.environ["WANDB_DIR"] = str(COLAB_LOCAL_DIR)
os.environ["PIP_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "pip-cache")
os.environ["TORCH_HOME"] = str(COLAB_LOCAL_DIR / "torch-cache")
print("Persistent output:", COLAB_OUTPUT_DIR)


In [ ]:
from importlib import metadata
import subprocess
import sys

# Preserve Colab's installed PyTorch/torchvision build during dependency resolution.
constraints = []
for package in ("torch", "torchvision"):
    try:
        constraints.append(package + "==" + metadata.version(package))
    except metadata.PackageNotFoundError:
        if True:
            raise RuntimeError("Use a Colab GPU runtime with PyTorch installed.") from None
constraint_file = COLAB_LOCAL_DIR / "colab-torch-constraints.txt"
constraint_file.write_text("\n".join(constraints), encoding="utf-8")
for distribution, package, version in [('ultralytics', 'ultralytics', '8.4.155'), ('wandb', 'wandb', '0.28.1'), ('PyYAML', 'PyYAML', None)]:
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or (version is not None and installed != version):
        requirement = package if version is None else package + "==" + version
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraint_file), requirement])


In [ ]:
"""Portable helpers embedded into each generated Colab notebook (no repo imports)."""

import hashlib
import os
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

import yaml


def colab_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def colab_api_key(name="WANDB_API_KEY"):
    """Prefer an existing environment value; otherwise request a Colab Secret."""
    value = os.environ.get(name, "").strip()
    if not value:
        try:
            from google.colab import userdata

            value = userdata.get(name)
        except Exception:  # noqa: BLE001 -- Normalize UI failures without exposing secret-bearing payloads.
            raise RuntimeError(
                f"Add {name} in Colab Secrets (key icon), enable notebook access, "
                "then rerun this cell. Never paste the key into notebook code."
            ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Colab Secret {name} is empty; update it and rerun this cell.")
    os.environ[name] = value.strip()
    return value.strip()


def colab_require_gpu(torch_module):
    if not torch_module.cuda.is_available():
        raise RuntimeError("Select Runtime > Change runtime type > GPU in Colab, then Run All.")


def colab_extract_zip(archive, destination):
    """Reject unsafe/duplicate members before extracting; ZIP CRCs validate reads."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as source:
        members = source.infolist()
        targets = []
        seen = set()
        for member in members:
            name = member.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if member.is_dir() and str(relative) == ".":
                continue  # Common `zip -r archive.zip .` root-directory entry.
            if (relative.is_absolute() or not relative.parts or ".." in relative.parts
                    or any(":" in part for part in relative.parts)
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {member.filename!r}")
            target = destination.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(destination) or target in seen:
                raise ValueError(f"Unsafe or duplicate ZIP member: {member.filename!r}")
            seen.add(target)
            targets.append((member, target))
        needed = sum(member.file_size for member in members)
        if needed > shutil.disk_usage(destination.parent).free:
            raise OSError("Insufficient Colab local disk for the extracted dataset.")
        destination.mkdir(parents=True, exist_ok=False)
        for member, target in targets:
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(member) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        return {"files": sum(not member.is_dir() for member in members), "bytes": needed}


def colab_resolve_reference(value, root, extracted, declared_root, source_declaration):
    """Relocate the declared root, keeping directory names and split membership."""
    value = str(value).replace("\\", "/")
    relative = PurePosixPath(value)
    if relative.is_absolute():
        prefixes = [str(declared_root or ""), str(PurePosixPath(source_declaration).parent)]
        for prefix in prefixes:
            prefix_path = PurePosixPath(prefix)
            if prefix and prefix_path.is_absolute() and relative.is_relative_to(prefix_path):
                relative = relative.relative_to(prefix_path)
                break
        else:
            raise ValueError(f"Cannot relocate absolute dataset path: {value}")
    candidate = root.joinpath(*relative.parts).resolve()
    # Same leading ../ compatibility used by the pinned Ultralytics loader.
    if not candidate.exists() and value.startswith("../"):
        candidate = (root / value[3:]).resolve()
    if not candidate.is_relative_to(extracted) or not candidate.exists():
        raise FileNotFoundError(f"Dataset path is missing or outside the extracted ZIP: {value}")
    return candidate


def prepare_colab_dataset(zip_path, local_directory, yaml_relative, source_declaration):
    """Copy a Drive ZIP locally and write a separate, fully resolved runtime YAML."""
    zip_path, local = Path(zip_path), Path(local_directory).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}. Edit DATASET_ZIP in Colab configuration.")
    local.mkdir(parents=True, exist_ok=True)
    runtime_yaml = local / "runtime/data.yaml"
    # A completed setup is immutable within this run; do not overwrite/reuse partial extraction.
    if (local / "source").exists() or runtime_yaml.exists():
        raise RuntimeError("Dataset setup already started for this run. Rerun the Colab configuration cell to create a new run directory.")
    archive = local / "dataset.zip"
    if zip_path.stat().st_size > shutil.disk_usage(local).free:
        raise OSError("Insufficient Colab local disk to copy DATASET_ZIP.")
    shutil.copyfile(zip_path, archive)
    archive_sha = colab_sha256(archive)
    if colab_sha256(zip_path) != archive_sha:
        raise ValueError("Dataset ZIP changed or was corrupted during the copy from Drive.")
    extracted = local / "source"
    inventory = colab_extract_zip(archive, extracted)
    if yaml_relative:
        original_yaml = (extracted / yaml_relative).resolve()
        if not original_yaml.is_relative_to(extracted) or not original_yaml.is_file():
            raise FileNotFoundError("DATASET_YAML_RELATIVE must name a YAML file inside the ZIP.")
    else:
        candidates = sorted(p for p in extracted.rglob("*") if p.suffix.lower() in {".yaml", ".yml"})
        if len(candidates) != 1:
            choices = [str(p.relative_to(extracted)) for p in candidates]
            raise ValueError(f"Expected one dataset YAML, found {choices}. Set DATASET_YAML_RELATIVE explicitly.")
        original_yaml = candidates[0]
    class UniqueDatasetLoader(yaml.SafeLoader):
        pass

    def unique_mapping(loader, node):
        mapping = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node)
            if key in mapping:
                raise ValueError(f"Duplicate dataset YAML key: {key!r}; fix the source ZIP explicitly.")
            mapping[key] = loader.construct_object(value_node)
        return mapping

    UniqueDatasetLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    raw = yaml.load(original_yaml.read_text(encoding="utf-8-sig"), Loader=UniqueDatasetLoader)
    if not isinstance(raw, dict) or not raw.get("train") or not raw.get("val"):
        raise ValueError("Dataset YAML must declare train, val and class names.")
    names = raw.get("names")
    if isinstance(names, dict):
        keys = sorted(names, key=lambda key: int(key))
        if [int(key) for key in keys] != list(range(len(keys))):
            raise ValueError("Class IDs must be consecutive starting at zero.")
        ordered_names = [str(names[key]) for key in keys]
    elif isinstance(names, list):
        ordered_names = [str(name) for name in names]
    else:
        raise TypeError("Dataset YAML names must be a list or a class-ID mapping.")
    if not ordered_names or int(raw.get("nc", len(ordered_names))) != len(ordered_names):
        raise ValueError("Dataset nc and class names disagree.")
    declared_root = raw.get("path")
    root = original_yaml.parent
    if declared_root and not PurePosixPath(str(declared_root).replace("\\", "/")).is_absolute():
        root = (root / declared_root).resolve()
    if not root.is_relative_to(extracted) or not root.is_dir():
        raise ValueError("Dataset YAML root must resolve inside the extracted ZIP.")
    relocated = dict(raw)
    relocated["path"] = str(root)
    runtime_yaml.parent.mkdir(parents=True, exist_ok=True)
    split_counts = {}
    # Match Ultralytics 8.4.155 IMG_FORMATS, including its less common formats.
    image_suffixes = {".avif", ".bmp", ".dng", ".heic", ".heif", ".jp2", ".jpeg", ".jpg", ".mpo", ".png", ".tif", ".tiff", ".webp"}
    for split in ("train", "val", "test"):
        if not raw.get(split):
            continue
        values = raw[split] if isinstance(raw[split], list) else [raw[split]]
        outputs, count = [], 0
        for index, value in enumerate(values):
            if not isinstance(value, str):
                raise TypeError(f"Invalid {split} path: {value!r}")
            path = colab_resolve_reference(value, root, extracted, declared_root, source_declaration)
            if path.is_dir():
                count += sum(p.is_file() and p.suffix.lower() in image_suffixes for p in path.rglob("*"))
                outputs.append(str(path))
            elif path.suffix.lower() == ".txt":
                entries = []
                for line in path.read_text(encoding="utf-8-sig").splitlines():
                    if not line.strip():
                        continue
                    image_root = path.parent if line.strip().startswith("./") else root
                    image = colab_resolve_reference(line.strip(), image_root, extracted, declared_root, source_declaration)
                    if not image.is_file() or image.suffix.lower() not in image_suffixes:
                        raise ValueError(f"Split list references a non-image: {line}")
                    entries.append(str(image))
                derived = runtime_yaml.parent / f"{split}-{index}.txt"
                derived.write_text("\n".join(entries) + "\n", encoding="utf-8")
                outputs.append(str(derived))
                count += len(entries)
            else:
                raise ValueError(f"Split must reference an image directory or TXT list: {value}")
        if count == 0:
            raise ValueError(f"No images found in {split}; check the ZIP and YAML paths.")
        relocated[split] = outputs if isinstance(raw[split], list) else outputs[0]
        split_counts[split] = count
    runtime_yaml.write_text(yaml.safe_dump(relocated, sort_keys=False, allow_unicode=True), encoding="utf-8")
    provenance = {
        "runtime": "google-colab", "drive_zip": str(zip_path), "zip_sha256": archive_sha,
        "extracted_inventory": inventory, "original_yaml": str(original_yaml),
        "original_yaml_sha256": colab_sha256(original_yaml), "runtime_yaml": str(runtime_yaml),
        "runtime_yaml_sha256": colab_sha256(runtime_yaml), "source_dataset_declaration": source_declaration,
        "class_names": ordered_names, "split_image_counts": split_counts,
        "relocation_only": True,
        "identity_note": "ZIP provenance is recorded; equality with the original Kaggle image/label bytes is not independently established.",
    }
    return str(runtime_yaml), provenance


In [ ]:
import json
import torch
colab_require_gpu(torch)
COLAB_DATA, COLAB_DATA_PROVENANCE = prepare_colab_dataset(
    DATASET_ZIP, COLAB_LOCAL_DIR, DATASET_YAML_RELATIVE, SOURCE_DATA_DECLARATION)
DATA = COLAB_DATA
(COLAB_OUTPUT_DIR / "dataset_provenance.json").write_text(
    json.dumps(COLAB_DATA_PROVENANCE, indent=2, ensure_ascii=False), encoding="utf-8")
print("Dataset:", COLAB_DATA)
print("Split image counts:", COLAB_DATA_PROVENANCE["split_image_counts"])


## 2. Environment Setup

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
import os

# Pertahankan instalasi PyTorch/CUDA bawaan Kaggle; instal hanya yang dibutuhkan.
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"Installing {package}=={version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import json
import time
from datetime import datetime
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

# Satu sumber logging W&B: callback eksplisit di section 6, bukan integrasi otomatis Ultralytics.
settings.update({"wandb": False})
init_seeds(SEED, deterministic=True)
RUNTIME_VERSIONS = {
    "python": sys.version.split()[0], "torch": torch.__version__,
    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb"),
    "cuda_runtime": torch.version.cuda,
}
import ultralytics
if ultralytics.__version__ != "8.4.155" or wandb.__version__ != "0.28.1":
    raise RuntimeError("Versi paket yang sudah diimport berbeda dari pin; restart kernel setelah instalasi.")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {metadata.version('ultralytics')} | W&B {metadata.version('wandb')}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU Colab atau set DEVICE='cpu' untuk test lokal.")

if WANDB_ENABLED:
    os.environ["WANDB_API_KEY"] = colab_api_key()  # Colab atau lokal: sediakan WANDB_API_KEY sebagai environment variable.
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("W&B online membutuhkan WANDB_API_KEY. Set sebagai Colab Secret atau environment variable; jangan tulis token ke notebook.")
    wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Dataset Loading

In [ ]:
from ultralytics.data.utils import check_det_dataset

# Memakai loader resmi, tanpa audit/cleaning dataset dalam notebook eksperimen.
data_info = check_det_dataset(DATA)
assert int(data_info["nc"]) > 0, "Dataset tidak memiliki kelas deteksi."
assert data_info.get("train") and data_info.get("val"), "Dataset wajib punya train dan val."
if EVAL_SPLIT != "val":
    raise ValueError("Protokol CSAKD menggunakan validation; test tidak boleh memilih checkpoint.")

# Periksa kesesuaian nama kelas teacher/dataset untuk metode KD.
def class_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    return [str(x) for x in names]

print(f"Dataset: {DATA} | classes: {data_info['nc']} | validation: {EVAL_SPLIT}")


## 4. Model Initialization

In [ ]:
import base64
import hashlib
import warnings

# Alias eksplisit v62; tidak beralih ke latest atau teacher pretrained.
api = wandb.Api()
artifact = api.artifact(TEACHER_ARTIFACT, type="model")
if artifact.version != "v62":
    raise ValueError(f"Versi teacher harus v62, diterima: {artifact.version}")
artifact_metadata = dict(artifact.metadata or {})
producer_info, producer_config, provenance_missing = {}, {}, []
try:
    producer = artifact.logged_by()
    if producer is not None:
        producer_info = {"id": producer.id, "name": producer.name, "path": list(producer.path)}
        producer_config = dict(producer.config or {})
        if producer.id != TEACHER_RUN_ID:
            raise ValueError(f"Producer teacher bukan run yang ditetapkan: {producer.id}")
    else:
        provenance_missing.append("producer")
except ValueError:
    raise
except (wandb.errors.Error, RuntimeError, AttributeError) as error:
    provenance_missing.append(f"producer unavailable: {type(error).__name__}")

if "best.pt" not in artifact.manifest.entries:
    raise ValueError("Manifest artifact teacher tidak berisi best.pt; file cache lokal tidak boleh menggantikannya.")
teacher_entry = artifact.manifest.entries["best.pt"]
print("[TEACHER] Mengunduh baseline YOLO26s v62...", flush=True)
artifact_dir = Path(artifact.download(root=str(COLAB_LOCAL_DIR / "teacher")))
teacher_path = artifact_dir / "best.pt"
if not teacher_path.is_file():
    raise FileNotFoundError(f"Artifact teacher tidak memiliki best.pt: {teacher_path}")
TEACHER_CHECKPOINT = str(teacher_path)
teacher_bytes = teacher_path.read_bytes()
teacher_sha256 = hashlib.sha256(teacher_bytes).hexdigest()
if teacher_entry.size is not None and len(teacher_bytes) != teacher_entry.size:
    raise ValueError("Ukuran best.pt berbeda dari manifest artifact.")
try:
    manifest_md5 = base64.b64decode(teacher_entry.digest, validate=True)
except (ValueError, TypeError):
    manifest_md5 = b""
if len(manifest_md5) == 16 and hashlib.md5(teacher_bytes).digest() != manifest_md5:
    raise ValueError("Checksum best.pt berbeda dari manifest artifact.")
del teacher_bytes

# Periksa checkpoint hasil eksperimen, kelas, dan skala arsitektur aktual.
teacher_meta = YOLO(str(teacher_path))
if int(data_info["nc"]) != EXPECTED_NC or int(teacher_meta.model.nc) != EXPECTED_NC:
    raise ValueError("Teacher dan Poltrada V3 harus memiliki enam kelas deteksi.")
if class_names(teacher_meta.model.names) != class_names(data_info["names"]):
    raise ValueError("Urutan nama kelas teacher dan dataset berbeda.")
architecture_yaml = str(teacher_meta.model.yaml.get("yaml_file", ""))
architecture_name = Path(architecture_yaml.replace("\\", "/").rsplit("/", 1)[-1]).stem
if architecture_name not in {"yolo26", "yolo26s"} or teacher_meta.model.yaml.get("scale") != "s":
    raise ValueError("Checkpoint teacher harus berasal dari keluarga YOLO26 dan memakai skala s.")
checkpoint_args = dict((teacher_meta.ckpt or {}).get("train_args") or {})

def verify_teacher_declarations(metadata_sources):
    """Tolak deklarasi berbeda; metadata kosong tidak dianggap verified."""
    declared_datasets, declared_models = {}, {}
    for source, values in metadata_sources.items():
        dataset = values.get("dataset", values.get("data"))
        if dataset:
            declared_datasets[source] = str(dataset)
            if str(dataset).replace("\\", "/") != SOURCE_DATA_DECLARATION.replace("\\", "/"):
                raise ValueError(f"Dataset teacher berbeda ({source}): {dataset}")
        architecture = values.get("model")
        if architecture:
            declared_models[source] = str(architecture)
            basename = str(architecture).replace("\\", "/").rsplit("/", 1)[-1]
            if Path(basename).stem != "yolo26s":
                raise ValueError(f"Arsitektur teacher berbeda ({source}): {architecture}")
    return declared_datasets, declared_models

teacher_datasets, teacher_models = verify_teacher_declarations({
    "artifact_metadata": artifact_metadata,
    "producer_config": producer_config,
    "checkpoint_train_args": checkpoint_args,
})
if not teacher_datasets:
    provenance_missing.append("declared dataset")
if not teacher_models:
    provenance_missing.append("declared model")
if not artifact.digest:
    provenance_missing.append("artifact digest")
TEACHER_PROVENANCE = {
    "requested_artifact": TEACHER_ARTIFACT, "resolved_name": artifact.name,
    "version": artifact.version, "digest": artifact.digest,
    "best_pt_manifest_digest": teacher_entry.digest,
    "producer": producer_info,
    "producer_config": {key: producer_config[key] for key in (
        "experiment", "model", "dataset", "seed", "epochs", "imgsz", "batch",
        "optimizer", "learning_rate", "amp") if key in producer_config},
    "artifact_metadata": artifact_metadata, "best_pt_sha256": teacher_sha256,
    "checkpoint_path": str(teacher_path), "declared_datasets": teacher_datasets,
    "declared_models": teacher_models, "nc": int(teacher_meta.model.nc),
    "names": class_names(teacher_meta.model.names), "architecture_scale": "s",
    "architecture_yaml": architecture_yaml,
    "metadata_complete": not provenance_missing, "missing_metadata": provenance_missing,
}
if provenance_missing:
    warnings.warn("Provenance teacher belum lengkap: " + "; ".join(provenance_missing))
    if not DIAGNOSTIC_ONLY:
        raise ValueError("Provenance teacher belum lengkap; lengkapi bukti source/dataset sebelum run penelitian.")
print("Teacher:", teacher_path, "| SHA256:", teacher_sha256)
print("Artifact:", artifact.name, "| version:", artifact.version, "| producer:", producer_info)
print("Epoch metadata (bukan versi artifact):", artifact_metadata.get("epoch"))
del teacher_meta

model = YOLO(MODEL_WEIGHTS)
print("[TEACHER] Verifikasi selesai; student:", MODEL_WEIGHTS, flush=True)


## 5. Knowledge Distillation Method

**Adaptasi CSAKD enam modul YOLO26s → YOLO26n, bukan reproduksi identik paper.** Tap [4, 6, 10, 16, 19, 22] berada pada backbone/neck. Input/output student diproyeksikan ke channel teacher; cross-forward menjalankan modul teacher yang frozen terhadap input student. Resolusi spasial wajib sama; tidak ada resize diam-diam. Target teacher asli disimpan melalui detach().clone() sebelum cross-forward.

Acuan primer: [paper BMVC2025, Zhan dkk.](https://bmva-archive.org.uk/bmvc/2025/assets/papers/Paper_398/paper.pdf) dan [kode loss resmi pada commit3433559](https://github.com/KefanZhan/YOLOv8-KD/blob/3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8/OtherKD/CSAKD.py).

**Formulasi numerik mengikuti kode penulis.** Dengan s output student terproyeksi, t snapshot teacher, c output cross, dan C channel: A=softmax(ReLU(c) @ ReLU(t).transpose(-1,-2) / sqrt(C), dim=-1) berbentuk **B×C×H×H**. AGKA=MSE(Norm_C((s+A@s)/2), Norm_C(t)). w=1-sigmoid(mean_B(sum_C(GAP(t)*GAP(s))/sqrt(C))); Cross=w*MSE(Norm_C(s), Norm_C(c)). Normalisasi L2 pada channel (eps=1e-12), MSE mean atas BCHW, KD FP32; jumlahkan keenam modul dengan bobot AGKA/Cross1/1.

**Perbedaan yang harus dilaporkan:** paper menyatakan Q dari teacher/K dari cross dan skala di luar softmax, sedangkan kode penulis menggunakan Q cross/K teacher dan skala sebelum softmax. Kode penulis menambahkan normalisasi channel/reduksi MSE mean. Notebook mengikuti kode untuk bagian numerik, dengan enam tap dan routing cross modul YOLO26 sebagai adaptasi (kode penulis memilih blok dalam cakupan layer berbeda). Paper membahas self-distillation YOLOv8 dan hybrid quantization; notebook memakai teacher YOLO26s/student YOLO26n tanpa quantization. Bobot1/1 tidak diklaim optimal paper.

Teacher tetap frozen/eval. Jalur teacher dari gambar tanpa gradien; cross-forward tetap berada dalam autograd agar student dan projector menerima gradien. Detection loss end-to-end YOLO26 (one-to-many/one-to-one beserta jadwal criterion) dipertahankan. KD mean dikalikan batch size tepat sekali; item logging dan rasio KD/detection tidak ditambahkan lagi ke objective.

**Strategi memori:** activation checkpoint non-reentrant pada kedua projector, cross teacher, attention, normalisasi, dan loss; chunk KD dua sampel. Student/detection tetap satu forward batch16. Mean AGKA, discrepancy, dan similarity chunk digabung berbobot ukuran chunk; sigmoid AAWM baru dihitung setelah similarity seluruh batch digabung. Semua komputasi KD tetap FP32. Capture hook teacher dinonaktifkan di dalam region, termasuk recomputation backward, lalu dipulihkan melalui finally. Target teacher disalin dan cache teacher dibersihkan sebelum forward student.

Strategi ini menukar komputasi dengan memori; belum merupakan bukti batch16/imgsz640 muat pada Tesla T4. Acuan: [checkpoint PyTorch2.11](https://docs.pytorch.org/docs/2.11/checkpoint.html), [Chen dkk., Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174).


In [ ]:
import math
from contextlib import contextmanager
from functools import partial
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.models.yolo.detect import DetectionTrainer


def detect_head(network):
    heads = [m for m in network.model if isinstance(m, Detect)]
    if len(heads) != 1:
        raise ValueError("Adaptasi KD ini mensyaratkan tepat satu Detect head.")
    return heads[0]


def align_channels(cin, cout):
    return nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)


def snapshot_student_checkpoint(source, destination):
    """Save a portable student copy; never strip the live training checkpoint."""
    from ultralytics.utils.torch_utils import strip_optimizer

    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        raise ValueError("Snapshot student harus berbeda dari checkpoint training asli.")
    if not source.is_file():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_data = strip_optimizer(str(source), str(destination))
    student = checkpoint_data.get("model")
    if not isinstance(student, nn.Module) or isinstance(student, DistillationModel):
        raise RuntimeError("Gagal menghasilkan checkpoint student-only yang portabel.")
    if hasattr(student, "teacher_model") or hasattr(student, "projector"):
        raise RuntimeError("Checkpoint inference masih membawa modul distillation.")
    for module in student.modules():
        hooks = list(module._forward_hooks.values()) + list(module._forward_pre_hooks.values())
        if any(isinstance(hook, (FeatureHook, InputHook)) for hook in hooks):
            raise RuntimeError("Checkpoint student masih membawa feature hook training.")
    if not destination.is_file():
        raise RuntimeError("File snapshot student tidak dibuat.")
    return destination


def combined_loss(wrapper, batch, preds, weights):
    """Native detection objective + KD mean * B; diagnostics never enter the objective."""
    wrapper._teacher_feats.clear()
    wrapper._student_feats.clear()
    try:
        if not wrapper.training:
            base, items = wrapper.student_model.loss(batch, preds)
            kd_parts = {key: base.new_zeros(()) for key in weights}
        else:
            with torch.no_grad():
                wrapper.teacher_model(batch["img"])
                # Keep only the six detached targets before allocating the student graph.
                targets = wrapper.snapshot_teacher_targets()
            predictions = wrapper.student_model(batch["img"])
            base, items = wrapper.student_model.loss(batch, predictions)
            with torch.autocast(device_type=batch["img"].device.type, enabled=False):
                kd_parts = wrapper.kd_loss(targets)
        kd = sum(weights[key] * kd_parts[key] for key in weights)
        if not torch.isfinite(base).all() or not all(torch.isfinite(value).all() for value in kd_parts.values()) or not torch.isfinite(kd).all():
            raise RuntimeError("Detection/KD loss non-finite; proses dihentikan.")
        batch_size = batch["img"].shape[0]
        detection_total = base.sum().detach() / batch_size
        items.update({key: value.detach() for key, value in kd_parts.items()})
        items["dis_loss"] = kd.detach()
        # E2ELoss native items report the one2one branch; this is the actual combined objective.
        items["det_total_loss"] = detection_total
        items["kd_to_det_ratio"] = kd.detach() / detection_total.abs().clamp_min(1e-12)
        return torch.cat((base.reshape(-1), (kd * batch_size).reshape(1))), items
    finally:
        wrapper._teacher_feats.clear()
        wrapper._student_feats.clear()


In [ ]:
def module_indices(network):
    # Six selected YOLO26 modules: backbone lateral outputs and neck Detect inputs.
    boundary = len(network.yaml["backbone"])
    lateral = set()
    for module in network.model[boundary:]:
        refs = module.f if isinstance(module.f, list) else [module.f]
        lateral.update(index for index in refs if 0 <= index < boundary)
    neck = list(detect_head(network).f)
    indices = sorted(lateral) + neck
    if indices != [4, 6, 10, 16, 19, 22]:
        raise ValueError("Adaptasi CSAKD ini memerlukan graph YOLO26 dengan tap [4, 6, 10, 16, 19, 22].")
    return indices


class ControlledFeatureHook(FeatureHook):
    """Native-compatible output hook with an instance-local teacher capture gate."""
    def __init__(self, cache, key, capture):
        super().__init__(cache, key)
        self.capture = capture

    def __call__(self, module, inputs, output):
        if self.capture["enabled"]:
            super().__call__(module, inputs, output)


class InputHook:
    def __init__(self, cache, key, capture=None):
        self.cache, self.key, self.capture = cache, key, capture

    def __call__(self, module, inputs):
        if self.capture is not None and not self.capture["enabled"]:
            return
        if len(inputs) != 1 or not isinstance(inputs[0], torch.Tensor):
            raise ValueError("Input modul CSAKD harus satu tensor.")
        self.cache[self.key] = inputs[0]


def csakd_loss_terms(student, teacher, cross):
    """Chunk-local means; AAWM sigmoid must be applied AFTER full-batch aggregation."""
    if student.ndim != 4 or student.shape != teacher.shape or teacher.shape != cross.shape:
        raise ValueError("Fitur CSAKD harus BCHW dan seukuran setelah channel projection.")
    s, t, c = student.float(), teacher.detach().float(), cross.float()
    scale = math.sqrt(s.shape[1])
    attention = torch.softmax(c.relu() @ t.relu().transpose(-2, -1) / scale, dim=-1)
    guidance = F.normalize((s + attention @ s) / 2, p=2, dim=1, eps=1e-12)
    target = F.normalize(t, p=2, dim=1, eps=1e-12)
    agka = F.mse_loss(guidance, target, reduction="mean")
    similarity = (t.mean((2, 3)) * s.mean((2, 3))).sum(dim=1) / scale
    discrepancy = F.mse_loss(
        F.normalize(s, p=2, dim=1, eps=1e-12),
        F.normalize(c, p=2, dim=1, eps=1e-12),
        reduction="mean",
    )
    return agka, discrepancy, similarity.mean()


def csakd_losses(student, teacher, cross):
    """Full-batch numerical loss from author commit 3433559; public return unchanged."""
    agka, discrepancy, similarity = csakd_loss_terms(student, teacher, cross)
    return agka, (1 - similarity.sigmoid()) * discrepancy


class CSAKDModel(DistillationModel):
    @staticmethod
    def get_distill_layers(network):
        return module_indices(network) + [detect_head(network).i]

    def _register_feature_hooks(self):
        # Fresh gate on construction/deepcopy/unpickle; never share state with another wrapper.
        self._teacher_capture = {"enabled": True}
        super()._register_feature_hooks()
        if self.teacher_model is not None:
            for index in self.feats_idx:
                module = self.teacher_model.model[index]
                for key, hook in list(module._forward_hooks.items()):
                    if isinstance(hook, FeatureHook):
                        # Preserve native RemovableHandle IDs and native hook cleanup support.
                        module._forward_hooks[key] = ControlledFeatureHook(self._teacher_feats, index, self._teacher_capture)
        for network, cache, handles in (
            (self.student_model, self._student_feats, self._student_hooks),
            (self.teacher_model, self._teacher_feats, self._teacher_hooks),
        ):
            if network is None:
                continue
            if self.feats_idx[:-1] != module_indices(network):
                raise ValueError("Index modul teacher/student CSAKD berbeda.")
            for index in self.feats_idx[:-1]:
                module = network.model[index]
                for key, hook in list(module._forward_pre_hooks.items()):
                    if isinstance(hook, InputHook):
                        del module._forward_pre_hooks[key]
                capture = self._teacher_capture if network is self.teacher_model else None
                handles.append(module.register_forward_pre_hook(InputHook(cache, ("input", index), capture)))

    def __init__(self, teacher_model, student_model, *, checkpoint_kd=None, chunk_size=None):
        checkpoint_flag = globals().get("KD_CHECKPOINT", True) if checkpoint_kd is None else checkpoint_kd
        chunk_limit = globals().get("KD_CHUNK_SIZE", 2) if chunk_size is None else chunk_size
        # Reject invalid configuration before registering hooks or loading teacher weights.
        if not isinstance(checkpoint_flag, bool):
            raise ValueError("KD_CHECKPOINT harus boolean.")
        if isinstance(chunk_limit, bool) or not isinstance(chunk_limit, int) or chunk_limit <= 0:
            raise ValueError("KD_CHUNK_SIZE harus integer positif.")
        super().__init__(teacher_model, student_model)
        self.kd_checkpoint, self.kd_chunk_size = checkpoint_flag, chunk_limit
        inputs, outputs = [], []
        for index in self.feats_idx[:-1]:
            sf, tf = self._student_feats[("input", index)], self._teacher_feats[("input", index)]
            so, to = self._student_feats[index], self._teacher_feats[index]
            if sf.shape[-2:] != tf.shape[-2:] or so.shape[-2:] != to.shape[-2:]:
                raise ValueError("Spatial feature teacher/student tidak seukuran.")
            inputs.append(align_channels(sf.shape[1], tf.shape[1]))
            outputs.append(align_channels(so.shape[1], to.shape[1]))
        self.projector = nn.ModuleDict({"input": nn.ModuleList(inputs), "output": nn.ModuleList(outputs)}).to(
            next(student_model.parameters()).device)
        self._teacher_feats.clear()
        self._student_feats.clear()

    def snapshot_teacher_targets(self):
        """Preserve original-image supervision and release teacher input/head caches early."""
        targets = {index: self._teacher_feats[index].detach().clone() for index in self.feats_idx[:-1]}
        self._teacher_feats.clear()
        return targets

    @contextmanager
    def _suspend_teacher_capture(self):
        previous = self._teacher_capture["enabled"]
        self._teacher_capture["enabled"] = False
        try:
            yield
        finally:
            self._teacher_capture["enabled"] = previous

    def _chunk_loss(self, student_raw, input_raw, target, *, input_projector, output_projector, teacher_module):
        # These contexts run again during backward recomputation, after combined_loss has returned.
        with self._suspend_teacher_capture(), torch.autocast(device_type=student_raw.device.type, enabled=False):
            student = output_projector(student_raw.float())
            incoming = input_projector(input_raw.float())
            cross = teacher_module(incoming)  # frozen weights, autograd to student/input adapter
            return csakd_loss_terms(student, target, cross)

    def kd_loss(self, targets=None):
        if targets is None:
            targets = self.snapshot_teacher_targets()
        components = []
        for pos, index in enumerate(self.feats_idx[:-1]):
            student_raw = self._student_feats[index]
            input_raw = self._student_feats[("input", index)]
            target = targets[index]
            if student_raw.shape[0] != target.shape[0] or input_raw.shape[0] != target.shape[0]:
                raise ValueError("Batch fitur student/input/teacher CSAKD berbeda.")
            batch_size = student_raw.shape[0]
            # Bind actual modules now; no late-bound loop variable or mutable feature-cache read.
            chunk_function = partial(
                self._chunk_loss, input_projector=self.projector["input"][pos],
                output_projector=self.projector["output"][pos], teacher_module=self.teacher_model.model[index],
            )
            agka = discrepancy = similarity = student_raw.new_zeros((), dtype=torch.float32)
            for start in range(0, batch_size, self.kd_chunk_size):
                end = min(start + self.kd_chunk_size, batch_size)
                arguments = (student_raw[start:end], input_raw[start:end], target[start:end])
                if self.kd_checkpoint and torch.is_grad_enabled():
                    terms = checkpoint(chunk_function, *arguments, use_reentrant=False, preserve_rng_state=True)
                else:
                    terms = chunk_function(*arguments)
                fraction = (end - start) / batch_size
                agka = agka + fraction * terms[0]
                discrepancy = discrepancy + fraction * terms[1]
                similarity = similarity + fraction * terms[2]
            # One AAWM weight over the ORIGINAL full batch; all three aggregates remain differentiable.
            components.append((agka, (1 - similarity.sigmoid()) * discrepancy))
        return {
            "kd_agka_loss": sum(x[0] for x in components),
            "kd_cross_loss": sum(x[1] for x in components),
        }

    def loss(self, batch, preds=None):
        return combined_loss(self, batch, preds, {"kd_agka_loss": KD_AGKA_WEIGHT, "kd_cross_loss": KD_CROSS_WEIGHT})


class KDTrainer(DetectionTrainer):
    def set_model_attributes(self):
        super().set_model_attributes()
        if self.args.compile or self.args.resume:
            raise ValueError("Adaptasi CSAKD: compile/resume tidak didukung.")
        self._requested_batch_size = self.args.batch
        self.model = CSAKDModel(self.args.distill_model, self.model)

    def _build_train_pipeline(self):
        # Ultralytics may halve batch after OOM; a changed protocol must stop, including retries.
        requested = getattr(self, "_requested_batch_size", self.args.batch)
        if self.batch_size != requested:
            raise RuntimeError(f"Batch CSAKD berubah dari {requested} ke {self.batch_size}; hentikan run dan evaluasi memori tanpa mengubah protokol diam-diam.")
        return super()._build_train_pipeline()

    def build_optimizer(self, model, name="auto", lr=0.001, momentum=0.9, decay=1e-5, iterations=1e5):
        optimizer = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        for group in optimizer.param_groups:
            group["params"] = [parameter for parameter in group["params"] if parameter.requires_grad]
        expected = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
        actual = {id(parameter) for group in optimizer.param_groups for parameter in group["params"]}
        if expected != actual:
            raise RuntimeError("Optimizer CSAKD tidak mencakup seluruh parameter trainable.")
        return optimizer

    def train(self):
        try:
            return super().train()
        finally:
            from ultralytics.utils.torch_utils import unwrap_model
            net = unwrap_model(self.model)
            if hasattr(net, "_remove_feature_hooks"):
                net._remove_feature_hooks()


KD_TRAINER = KDTrainer
KD_TRAIN_ARGS = {"distill_model": str(teacher_path), "dis": 1.0, "compile": False}
KD_CONFIG = {
    "agka_weight": KD_AGKA_WEIGHT, "cross_weight": KD_CROSS_WEIGHT,
    "weight_policy": "fixed-study-adaptation-1-1", "taps": [4, 6, 10, 16, 19, 22],
    "loss_reference_commit": "3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8",
    "loss_formulation": "author-normalized-mse-row-attention",
    "feature_normalization": {"p": 2, "dim": 1, "eps": 1e-12},
    "attention_shape": "B,C,H,H", "cross_routing": "six-local-student-input-modules",
    "kd_precision": "float32", "batch_scaling": "mean-times-batch-once",
    "memory_strategy": {"activation_checkpoint": KD_CHECKPOINT, "use_reentrant": False,
                        "preserve_rng_state": True, "kd_chunk_size": KD_CHUNK_SIZE,
                        "student_batch_unchanged": True, "aawm_scope": "full-batch-before-sigmoid"},
}


## 6. Model Training

In [ ]:
from datetime import datetime

RUN_NAME = f"{EXPERIMENT_ID}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}"
train_args = dict(
    data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    project=PROJECT, name=RUN_NAME, exist_ok=False,
    workers=WORKERS, patience=PATIENCE, plots=True,
    amp=AMP, pretrained=True, save=True,
)
train_args.update(KD_TRAIN_ARGS)

# W&B: satu run dan satu koleksi artifact checkpoint berversi per eksperimen.
wb_run = None
wb_state = {"uploaded_epochs": set(), "logged_epochs": set(), "best_epoch": None}
effective_protocol = {}

def record_effective_protocol(trainer):
    """Catat protokol aktual dan hentikan perubahan batch atau penonaktifan CUDA AMP."""
    actual_batch = int(trainer.batch_size)
    amp_active = bool(trainer.amp)
    effective_protocol.update({
        "requested_batch": int(BATCH), "effective_batch": actual_batch,
        "per_rank_loader_batch": int(trainer.train_loader.batch_size),
        "world_size": max(int(trainer.world_size), 1),
        "requested_amp": bool(AMP), "effective_amp": amp_active,
        "device_type": trainer.device.type,
        "diagnostic_only": bool(DIAGNOSTIC_ONLY),
    })
    if actual_batch != int(BATCH):
        raise RuntimeError("Batch efektif berubah; run dihentikan agar protokol penelitian tetap konsisten.")
    if not DIAGNOSTIC_ONLY and AMP and trainer.device.type == "cuda" and not amp_active:
        raise RuntimeError("CUDA AMP yang diminta tidak aktif; periksa kompatibilitas sebelum eksperimen final.")
    if wb_run is not None:
        wb_run.config.update({"effective_protocol": dict(effective_protocol)}, allow_val_change=True)

model.add_callback("on_train_start", record_effective_protocol)
model.add_callback("on_train_epoch_start", record_effective_protocol)
if WANDB_ENABLED:
    wb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=WANDB_RUN_NAME,
        job_type="training",
        group=f"diagnostic-{WANDB_GROUP}" if DIAGNOSTIC_ONLY else WANDB_GROUP,
        config={**({
            "experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if TEACHER_CHECKPOINT else None,
            "dataset": DATA, "eval_split": EVAL_SPLIT, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": str(DEVICE),
            "optimizer": OPTIMIZER, "learning_rate": LR, "amp": AMP,
            "workers": WORKERS, "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd": KD_CONFIG, "diagnostic_only": DIAGNOSTIC_ONLY,
            "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
            "csakd_author_commit": CSAKD_AUTHOR_COMMIT,
        }), "colab_dataset": COLAB_DATA_PROVENANCE, "runtime": "google-colab"},
        tags=[*([*WANDB_TAGS, "diagnostic" if DIAGNOSTIC_ONLY else "research"]), "colab"],
    )

    wb_run.use_artifact(artifact)

    def save_epoch_artifact(trainer):
        """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
        epoch = int(trainer.epoch) + 1
        if epoch in wb_state["uploaded_epochs"]:
            return
        last_path = Path(trainer.last)
        best_path = Path(trainer.best)
        if not last_path.is_file():
            raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
        fitness = None if trainer.fitness is None else float(trainer.fitness)
        best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
        is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
        if is_best:
            wb_state["best_epoch"] = epoch
        artifact = wandb.Artifact(
            name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
            type="model",
            description="Student-only best.pt/last.pt; checkpoint asli training_state_* memerlukan class notebook (resume tidak didukung).",
            metadata={
                "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
                "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
                "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
                "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
                "kd": KD_CONFIG, "effective_protocol": dict(effective_protocol),
                "diagnostic_only": bool(DIAGNOSTIC_ONLY),
            },
        )
        snapshot_dir = Path(trainer.save_dir) / "student_checkpoints"
        snapshot_dir.mkdir(parents=True, exist_ok=True)
        portable_last = snapshot_student_checkpoint(last_path, snapshot_dir / "last.pt")
        artifact.add_file(str(portable_last), name="last.pt")
        artifact.add_file(str(last_path), name="training_state_last.pt")
        if best_path.is_file():
            portable_best = snapshot_student_checkpoint(best_path, snapshot_dir / "best.pt")
            artifact.add_file(str(portable_best), name="best.pt")
            artifact.add_file(str(best_path), name="training_state_best.pt")
        for filename in ("results.csv", "args.yaml"):
            file_path = Path(trainer.save_dir) / filename
            if file_path.is_file():
                artifact.add_file(str(file_path), name=filename)
        aliases = [f"epoch-{epoch:03d}", "latest"]
        if is_best:
            aliases.append("best")
        # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
        wb_run.log_artifact(artifact, aliases=aliases).wait()
        wb_state["uploaded_epochs"].add(epoch)
        print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

    def log_epoch_metrics(trainer):
        """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
        epoch = int(trainer.epoch) + 1
        # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
        if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
            return
        loss = trainer.label_loss_items(trainer.tloss, prefix="train")
        payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
                   **{k: float(v) for k, v in trainer.lr.items()}}
        for key, value in trainer.metrics.items():
            try:
                payload[key] = float(value)
            except (ValueError, TypeError):
                continue
        payload["protocol/effective_batch"] = int(trainer.batch_size)
        payload["protocol/effective_amp"] = int(bool(trainer.amp))
        if trainer.device.type == "cuda":
            payload["memory/peak_allocated_gb"] = float(torch.cuda.max_memory_allocated(trainer.device) / 1024**3)
            payload["memory/peak_reserved_gb"] = float(torch.cuda.max_memory_reserved(trainer.device) / 1024**3)
        payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
        payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
        wb_run.log(payload, step=epoch)
        wb_state["logged_epochs"].add(epoch)

    model.add_callback("on_model_save", save_epoch_artifact)
    model.add_callback("on_fit_epoch_end", log_epoch_metrics)

print("Training:", EXPERIMENT_ID, "| data:", DATA, "| epochs:", EPOCHS,
      "| W&B:", "online + every epoch artifact" if WANDB_ENABLED else "disabled")
start_time = time.perf_counter()
try:
    if KD_TRAINER is None:
        model.train(**train_args)
    else:
        model.train(trainer=KD_TRAINER, **train_args)
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise
training_seconds = time.perf_counter() - start_time
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best_pt}")
if wb_run is not None and len(wb_state["uploaded_epochs"]) != EPOCHS:
    print("Catatan: jumlah epoch selesai berbeda dari target (misalnya early stopping).",
          len(wb_state["uploaded_epochs"]), "/", EPOCHS)
print("Training selesai:", run_dir)


## 7. Validation & Reloaded Student Inference


In [ ]:
try:
    # Salinan student-only tidak mengubah checkpoint training asli.
    deployment_best_pt = snapshot_student_checkpoint(best_pt, run_dir / "student_checkpoints" / "best.pt")
    deployment_last_pt = snapshot_student_checkpoint(run_dir / "weights" / "last.pt", run_dir / "student_checkpoints" / "last.pt")
    best_model = YOLO(str(deployment_best_pt))
    metrics = best_model.val(
        data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, plots=True, verbose=False,
    )

    # Inference checkpoint reload pada satu gambar validation; box kosong tetap valid.
    inference_source = model.trainer.test_loader.dataset.im_files[0]
    reload_predictions = best_model.predict(source=inference_source, imgsz=IMGSZ, device=DEVICE, verbose=False)
    if len(reload_predictions) != 1 or reload_predictions[0].boxes is None:
        raise RuntimeError("Inferensi student reload tidak menghasilkan struktur deteksi yang valid.")
    if not torch.isfinite(reload_predictions[0].boxes.data).all():
        raise RuntimeError("Prediksi student reload mengandung NaN/Inf.")
    print("Validation selesai; inference student reload valid:", inference_source)

    from ultralytics.utils.torch_utils import get_flops
    parameters = sum(p.numel() for p in best_model.model.parameters())
    try:
        gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
    except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
        print("GFLOPs tidak tersedia:", error)
        gflops = None

    results = {
        "experiment": EXPERIMENT_ID,
        "method": METHOD,
        "model": MODEL_WEIGHTS,
        "dataset": DATA,
        "eval_split": EVAL_SPLIT,
        "evaluation_label": "validation", "diagnostic_only": bool(DIAGNOSTIC_ONLY),
        "seed": SEED,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "optimizer": OPTIMIZER,
        "learning_rate": LR,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "parameters": int(parameters),
        "gflops": gflops,
        "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
        "training_seconds": float(training_seconds),
        "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
        "best_pt": str(best_pt), "deployment_best_pt": str(deployment_best_pt),
        "deployment_last_pt": str(deployment_last_pt),
        "reload_inference_valid": True, "reload_inference_source": str(inference_source),
        "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
        "effective_protocol": dict(effective_protocol),
    }
    if torch.cuda.is_available():
        results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
        results["peak_gpu_reserved_gb"] = float(torch.cuda.max_memory_reserved() / 1024**3)
    print(json.dumps(results, indent=2, ensure_ascii=False))
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise


## 8. Results & Outputs

In [ ]:
# File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
run_dir.mkdir(parents=True, exist_ok=True)
results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
(run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
configuration = {
    "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
    "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
    "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
    "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "close_mosaic": CLOSE_MOSAIC,
    "kd_hyperparameters": KD_CONFIG, "csakd_author_commit": CSAKD_AUTHOR_COMMIT,
    "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
    "effective_protocol": dict(effective_protocol), "diagnostic_only": bool(DIAGNOSTIC_ONLY),
    "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
}
(run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

if wb_run is not None:
    summary = {f"final/{key}": val for key, val in results.items()
               if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
    wb_run.log(summary)
    wb_run.summary.update(summary)
    wb_run.summary["final/evaluation_split"] = EVAL_SPLIT
    wb_run.summary["final/evaluation_label"] = "validation"
    wb_run.summary["teacher_provenance_complete"] = TEACHER_PROVENANCE["metadata_complete"]
    wb_run.summary["best_epoch"] = wb_state["best_epoch"]
    wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
    wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

    # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
    final_artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
        metadata={"experiment": EXPERIMENT_ID, "best_epoch": wb_state["best_epoch"]},
    )
    for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
        candidate = run_dir / filename
        if candidate.is_file():
            final_artifact.add_file(str(candidate), name=filename)
    for folder_name, folder in (("training", run_dir),
                                ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
        if folder.exists():
            for plot_path in sorted(folder.glob("*.png")):
                key = f"plots/{folder_name}/{plot_path.stem}"
                wb_run.log({key: wandb.Image(str(plot_path))})
                final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
    wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
    print("W&B:", wb_run.url)
    print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))
    wb_run.finish()

print("Hasil lokal:", run_dir)
print("  - weights/best.pt, weights/last.pt (student-only setelah final_eval Ultralytics)")
if WANDB_ENABLED:
    print("  - Artifact per epoch: training_state_best.pt/last.pt (state asli, class notebook diperlukan)")
print("  - student_checkpoints/best.pt, last.pt (student-only untuk inferensi)")
print("  - results.csv, args.yaml")
print("  - metrics.json, config.json")
